In [291]:
import numpy as np
from numpy.linalg import eig

In [292]:
class HMM:
    def __init__(self, transitions, emissions):

        self.A = np.array([list(state.values()) for state in transitions.values()]) #transition matrix
        self.B = np.array([list(state.values()) for state in emissions.values()]) #emission matrix

        eigenvalues, eigenvectors = np.linalg.eig(self.A.T)
        idx = np.argmin(np.abs(eigenvalues - 1))
        s_dist = eigenvectors[:, idx].real
        self.pi = s_dist / s_dist.sum() #normalised stationary distribution

        self.states = list(transitions.keys()) #list of hidden states
        self.observations = list(next(iter(emissions.values())).keys()) #list of observable outputs

In [293]:
def forward_algorithm(model, observed_seq):
    A = model.A
    B = model.B
    pi = model.pi
    states = model.states
    observations = model.observations

    n_states = len(states)
    obs_to_idx = {o: i for i, o in enumerate(observations)}
    obs_indices = [obs_to_idx[o] for o in observed_seq]

    #we can just use alphas_old, alphas_new, because we don't need to store past alpha_t values once alpha_t+1 values have been calculated 
    #already computed for first time step: when i = 0
    alphas_old = [ pi[j] * B[j][obs_indices[0]] for j in range(n_states)]

    #computing for other time steps
    for time in range(1, len(observed_seq)):
        alphas_new = [0.0]*n_states
        obs_idx = obs_indices[time]

        #calculating alpha_time(state x) =
        #sum of (alpha_time-1 for all states * probability of going from that state to state x )  * emission probability of observance with state x
        for x in range(n_states):
            alphas_new[x] = sum(alphas_old[k]* A[k][x] for k in range(n_states)) * B[x][obs_idx]
        alphas_old = alphas_new

    print(f"The probability of getting this sequence of observations is {sum(alphas_old)}")

In [294]:
def forward_backward_algorithm(model, observed_seq, t="", hidden_s = ""):
    #Fetching our matrices, list of states/observations, number of states, length of sequence
    A = model.A
    B = model.B
    pi = model.pi
    states = model.states
    observations = model.observations

    num_states = len(states)
    length = len(observed_seq)
    states_to_idx = {s: i for i, s in enumerate(states)}
    obs_to_idx = {o: i for i, o in enumerate(observations)}
    obs_indices = [obs_to_idx[o] for o in observed_seq]

    alphas = [  [[0.0] for _ in range(num_states)]  for _ in range(length)]
    betas = [ [[0.0] for _ in range(num_states)] for _ in range(length)]

    #FORWARD ALGO
    for x in range(num_states): alphas[0][x] = pi[x]*B[x][obs_indices[0]] #already computed for time=0
    #computing for other time steps
    for time in range(1, length):
        obs_idx = obs_indices[time]
        #calculating alpha_time(state x) = sum of (alpha_time-1 for all states * probability of going from that state to state x )  * emission probability of observance with state x
        for x in range(num_states):
            alphas[time][x] = sum(alphas[time-1][k]* A[k][x] for k in range(num_states)) * B[x][obs_idx]
    
    #BACKWARD ALGO
    for x in range(num_states-1, -1, -1): betas[-1][x] = 1 #initialising Beta_time-end(all states) = 1
    for time in range(length-2,-1,-1):
        obs_idx = obs_indices[time+1]
        for x in range(num_states):
            betas[time][x] = sum(betas[time+1][k] * A[x][k] * B[k][obs_idx] for k in range(num_states))
    
    gammas = [ [0.0 for _ in range(num_states)] for _ in range(length)]
    for time in range(length):
        normalisation = sum(alphas[time][k] * betas[time][k] for k in range(num_states))
        for x in range(num_states): gammas[time][x] = (alphas[time][x] * betas[time][x]) / normalisation

    if str(t).isnumeric() and hidden_s in states:
        print(f"The probability of getting State {hidden_s} at Time {t} is {gammas[t-1][states_to_idx[hidden_s]]}")
    elif str(t).isnumeric()==True and str(hidden_s).isnumeric() and hidden_s > -1 and hidden_s < len(observations):
        print(f"The probability of getting State {hidden_s} at Time {t} is {gammas[t][hidden_s]}")
    else: print("\n".join( f"Time {ti+1}: {gamma_t}, Sum: {sum(gamma_t)} " for ti, gamma_t in enumerate(gammas)) )
    


In [295]:
def viterbi_algorithm(model, observed_seq):
    #Fetching our matrices, list of states/observations, number of states, length of sequence
    A = model.A
    B = model.B
    pi = model.pi
    states = model.states
    observations = model.observations
    num_states = len(states)
    length = len(observed_seq)

    #Giving every observation output a specific number/index; mapping observed sequence to a sequence of numbers/indices.
    obs_to_idx = {o: i for i, o in enumerate(observations)}
    obs_indices = [obs_to_idx[o] for o in observed_seq]


    #storing probs_time(X) and most likely path to get to X from a previous state
    probs = [  [[0.0] for _ in range(num_states)]  for _ in range(length)]
    backtr = [ [[0] for _ in range(num_states)] for _ in range(length)]

    #Initialisation
    for x in range(num_states):
        probs[0][x] = pi[x] * B[x][obs_indices[0]]
        backtr[0][x] = None

    #Dynamic Programming
    for time in range(1, length):
        observed_idx = obs_indices[time]

        for x in range(num_states):
            #Finding the probability, of a path from a previous_state in Time-1 to state X at Time
            #Storing the path with max probability to X: probs in probs[time][x] and the previous state in backtr[time][x]
            max_p = -1.0
            arg_max = None

            for prev_state in range(num_states):
                term = probs[time-1][prev_state] * A[prev_state][x]
                if term > max_p:
                    max_p = term
                    arg_max = prev_state

            probs[time][x] = max_p * B[x][observed_idx] #multiplying everything through by P(observed | state) later
            backtr[time][x] = arg_max
    
    #Backtracking
    idx_seq = [0]*length
    seq = [""]*length

    #Find most likely hidden state on last day
    last = max(range(num_states), key=lambda x: probs[length-1][x])
    idx_seq[length-1] = last
    #working backwards to find most likely sequence
    for time in range(length-2, -1, -1):
        idx_seq[time] = backtr[time+1][last]
        last = backtr[time+1][last]
    
    #producing sequence, from idx_seq
    for time, idx in enumerate(idx_seq):
        seq[time] = states[idx]
    print(f"The most probable sequence of hidden states is {seq}")


Given an observed sequence:

Forward Algorithm gives probability of being in State X at Time i ( alpha_time(X) ), given all observations from Time 0 to Time i.
If you sum alpha_final-time (all states) you get the overall probability of the observed sequence. (returned as a decimal)

Forward-Backward Algorithm gives probability of being in State X at Time i, given all observations in the sequence. Combination of alpha_time(X) and beta_time(X), using forward and backwards passes.

Viterbi Algorithm gives most probable sequence of hidden States (returned as a sequence)

In [ ]:
transitions ={
    "sunny": {"sunny": 0.6, "rainy": 0.3, "cloudy": 0.1},
    "rainy": {"sunny": 0.1, "rainy": 0.5, "cloudy": 0.4},
    "cloudy": {"sunny": 0.3, "rainy": 0.4, "cloudy": 0.3},
}
emissions = {
    "sunny": {"happy": 0.8, "sad": 0.1, "neutral": 0.1},
    "rainy": {"happy": 0.2, "sad": 0.6, "neutral": 0.2},
    "cloudy": {"happy": 0.4, "sad": 0.4, "neutral": 0.2},
}
model_1 = HMM(transitions, emissions)

observed_seq = ["sad", "sad", "happy"]

#this functions returns how likely the provided observed sequence is
#if we actually generated a full array of alphas instead of discarding old alphas values, it could return probability of any hidden state at Time T, given observed sequence up to Time T
forward_algorithm(model_1, observed_seq)

#gives likelihood of a hidden_state at some point in Time in the observed sequence
forward_backward_algorithm(model_1, observed_seq, t = 3, hidden_s="sunny") 

#viterbi tells you the most likely state sequence which corresponds to the provided observed sequence
viterbi_algorithm(model_1, observed_seq) 

The probability of getting this sequence of observations is 0.06428
The probability of getting State sunny at Time 3 is 0.3908719026390689
The most probable sequence of hidden states is ['rainy', 'rainy', 'cloudy']
